# SC203 DoSSR — Phase-2 LoRA Sweep, Timestep Gating, Base DoSSR Validation & 2,000-Step Curve (`04_lora_sweep_and_2000step_colab.ipynb`)

**Team:** LEMMEPASS (Team 1, SC203 Scientific Methods — Class 25A02)  
**Target Branch:** `feat/baseline-runner`  
**Mentor Directives Implemented (Dr. Thuận, Nov 20, 2026):**
1. **Math & Gradient Verification:** `src/lora_text_loss.py --run-cpu-tests --verify-fp32-gradients --audit-eta-schedule` + optional real-model FP32 two-pass vs. joint backward audit (`--verify-fp32-grad`).
2. **Real-CE 945-Pair Provenance & Scene Split IDs:** Verify `945 = 645 (official train) + 300 (official val)` and export full train/val scene IDs (`data/realce_splits_manifest.json`).
3. **Base DoSSR + Bicubic $\times 4$ on Exact Same 30 Val Patches:** Evaluate unmodified base DoSSR (`per_image_seed: true`, `42..51`) and Bicubic whole-patch LPIPS on `RealCE_patches/val`.
4. **$\lambda_{\text{cons}} \in \{0.1, 0.2\}$ 400-Step Sweep:** Train 400 optimizer steps (`grad_accum=4` $\implies 1{,}600$ minibatches) with identical `seed=231`, init, and batch order to Pilot `0.0` & `0.5`.
5. **Timestep-Gated Consistency Ablation ($t \le 499$, matching `trunc_12` $t_1 = 500$):** Compare full $t \in [0, 999]$ vs. $t \in [0, 499]$ using selected $\lambda^*$, logging sampled $t$ and active consistency ratio.
6. **2,000-Step Extension (`8,000` minibatches, `save_every=400`) + Checkpoint Validation Curves & Optional 2nd Seed (`seed=777`).**

## CELL 01 — Mount Drive, Pull Latest Branch, Audit `eta_t` & FP32 Two-Pass Gradients

In [ ]:
# === CELL 01: Mount Drive, pull SC203, check L4 GPU, & run math/FP32 gradient audits ===
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = Path('/content/SC203')
DRIVE_BASE = Path('/content/drive/MyDrive/lemmepass')
DATA_DIR = DRIVE_BASE / 'data'
OUT_BASE = DRIVE_BASE / 'outputs'
CKPT_PATH = DRIVE_BASE / 'weights' / 'dossr_default.ckpt'
BRANCH = 'feat/baseline-runner'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/lemon4life/SC203.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)

dossr_cfg = REPO_DIR / 'external' / 'DOSSR' / 'configs' / 'model' / 'cldm_v21.yaml'
REQUIRED_DOSSR_SPECS = ('pytorch_lightning', 'open_clip', 'omegaconf', 'einops', 'transformers', 'timm')
missing_dossr = [pkg for pkg in REQUIRED_DOSSR_SPECS if importlib.util.find_spec(pkg) is None]
if not dossr_cfg.is_file() or missing_dossr:
    if importlib.util.find_spec('jedi') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'jedi==0.20.1'], check=False)
    subprocess.run([sys.executable, '-u', 'scripts/colab_setup.py', '--allow-preexisting-conflicts'], cwd=REPO_DIR, check=True)

missing_pkgs = [pkg for pkg in ('rapidocr_onnxruntime', 'lpips') if importlib.util.find_spec(pkg) is None]
if missing_pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing_pkgs], check=True)

def run_streamed(cmd):
    print('RUN:', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'Command failed with exit code {rc}')

import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.2f} GiB')

run_streamed([sys.executable, '-u', 'src/lora_text_loss.py', '--run-cpu-tests', '--verify-fp32-gradients', '--audit-eta-schedule'])
run_streamed([sys.executable, '-u', 'scripts/run_lora_trial.py', '--dry-run-cpu'])

## CELL 02 — Stage Real-CE, Verify 945 = 645 `train` + 300 `val`, & Export Exact Split IDs

In [ ]:
# === CELL 02: Stage Real-CE on fast local SSD, verify 645/300 counts, & extract identical train/val patches ===
import json
import zipfile

sys.path.insert(0, str(REPO_DIR))
from scripts.check_realce_feasibility import resolve_realce_root
from scripts.split_realce_by_scene import discover_paired_samples

LOCAL_REALCE = Path('/content/data_local/Real-CE')
DRIVE_REALCE = DATA_DIR / 'Real-CE'
ZIP_CANDIDATES = [Path('/content/Real-CE.zip'), DATA_DIR / 'Real-CE.zip']

def _has_realce(path: Path) -> bool:
    r = resolve_realce_root(path)
    return (r / 'train' / '13mm').is_dir() and (r / 'train' / '52mm').is_dir()

if _has_realce(LOCAL_REALCE):
    REALCE_DIR = resolve_realce_root(LOCAL_REALCE)
elif any(z.is_file() for z in ZIP_CANDIDATES):
    src_zip = next(z for z in ZIP_CANDIDATES if z.is_file())
    LOCAL_REALCE.mkdir(parents=True, exist_ok=True)
    rc = subprocess.run(['unzip', '-q', '-o', str(src_zip), '-d', str(LOCAL_REALCE)], check=False).returncode
    if rc != 0:
        with zipfile.ZipFile(src_zip, 'r') as zf:
            zf.extractall(LOCAL_REALCE)
    REALCE_DIR = resolve_realce_root(LOCAL_REALCE)
else:
    REALCE_DIR = resolve_realce_root(DRIVE_REALCE)

official_train = discover_paired_samples(REALCE_DIR, split='train', scale='x4')
official_val = discover_paired_samples(REALCE_DIR, split='val', scale='x4')
print(f'Official Real-CE 4x counts -> train: {len(official_train)} | official val (untouched): {len(official_val)} | total: {len(official_train) + len(official_val)}')

REALCE_PATCH_DIR = Path('/content/data_local/RealCE_patches')
run_streamed([
    sys.executable, '-u', 'scripts/split_realce_by_scene.py',
    '--data-dir', str(REALCE_DIR),
    '--split', 'train',
    '--scale', 'x4',
    '--val-ratio', '0.20',
    '--seed', '42',
    '--output-manifest', 'data/realce_splits_manifest.json',
    '--output-train-csv', 'data/realce_train_manifest.csv',
    '--output-train-boxes', 'data/realce_train_boxes.csv',
    '--output-val-boxes', 'data/realce_val_boxes.csv',
    '--extract-patches',
    '--patch-dir', str(REALCE_PATCH_DIR),
    '--hr-patch-size', '512',
    '--lr-patch-size', '128',
    '--stride', '256',
    '--max-patches-per-scene', '2',
    '--max-val-patches', '30',
])

manifest = json.loads(Path('data/realce_splits_manifest.json').read_text(encoding='utf-8'))
overlap = set(manifest['train_scenes']) & set(manifest['val_scenes'])
print(f"Train scenes: {len(manifest['train_scenes'])} | Val scenes: {len(manifest['val_scenes'])} | Scene overlap: {len(overlap)}")
print('First 10 train scenes:', manifest['train_scenes'][:10])
print('First 10 val scenes  :', manifest['val_scenes'][:10])

# Copy full split manifest to Drive so team has the exact train/val scene ID list archived
(DRIVE_BASE / 'reports').mkdir(parents=True, exist_ok=True)
(DRIVE_BASE / 'reports' / 'realce_splits_manifest_populated.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

## CELL 03 — Evaluate Unmodified Base DoSSR (No LoRA) & Bicubic $\times 4$ Whole-Patch Metrics on Exact Same 30 Val Patches

In [ ]:
# === CELL 03: Shared evaluation helper + Bicubic whole-patch LPIPS + Base DoSSR 10-seed val evaluation ===
import yaml

VAL_RESULTS_DIR = DRIVE_BASE / 'results_phase2'
VAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def write_val_cfg(src_yaml: Path, dst_yaml: Path, *, out_root: Path, lora_weights: Path | None = None) -> Path:
    cfg = yaml.safe_load(src_yaml.read_text(encoding='utf-8'))
    cfg['input_dir'] = str(REALCE_PATCH_DIR / 'val' / 'LR')
    cfg['gt_dir'] = str(REALCE_PATCH_DIR / 'val' / 'HR')
    cfg['output_root'] = str(out_root)
    if lora_weights is not None:
        cfg['lora_weights'] = str(lora_weights)
    else:
        cfg.pop('lora_weights', None)
    cfg['baseline_config'] = str(dst_yaml)
    dst_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    return dst_yaml

def eval_study_on_val(tag: str, src_yaml: Path, *, study_out: Path, lora_weights: Path | None = None) -> dict:
    runtime_yaml = Path(f'/content/val_cfg_{tag}.yaml')
    write_val_cfg(src_yaml, runtime_yaml, out_root=study_out, lora_weights=lora_weights)
    run_streamed([sys.executable, '-u', 'scripts/run_seed_study.py', '--config', str(runtime_yaml)])

    metrics_sum = VAL_RESULTS_DIR / f'metrics_{tag}_val_summary.csv'
    metrics_per_seed = VAL_RESULTS_DIR / f'metrics_{tag}_val_per_seed.csv'
    run_streamed([
        sys.executable, '-u', 'scripts/compute_metrics.py',
        '--output-root', str(study_out),
        '--gt-dir', str(REALCE_PATCH_DIR / 'val' / 'HR'),
        '--expected-size', '512',
        '--test-y-channel',
        '--device', 'cuda',
        '--summary-csv', str(metrics_sum),
        '--per-image-csv', str(metrics_per_seed),
    ])

    ocr_dir = VAL_RESULTS_DIR / f'ocr_{tag}_reconly'
    run_streamed([
        sys.executable, '-u', 'scripts/text_eval.py',
        '--study-root', str(study_out),
        '--boxes-csv', 'data/realce_val_boxes.csv',
        '--ocr-backend', 'rapidocr',
        '--rec-only',
        '--rotate-vertical',
        '--upscale', '2',
        '--output-dir', str(ocr_dir),
    ])
    return {'tag': tag, 'study_out': study_out, 'metrics_summary': metrics_sum, 'ocr_dir': ocr_dir}

# 1. Whole-patch Bicubic x4 LPIPS/PSNR/SSIM on Real-CE Val (30 patches)
BICUBIC_VAL_SUMMARY = VAL_RESULTS_DIR / 'metrics_bicubic_realce_val_summary.csv'
run_streamed([
    sys.executable, '-u', 'scripts/compute_metrics.py',
    '--bicubic-lr-dir', str(REALCE_PATCH_DIR / 'val' / 'LR'),
    '--gt-dir', str(REALCE_PATCH_DIR / 'val' / 'HR'),
    '--expected-size', '512',
    '--test-y-channel',
    '--device', 'cuda',
    '--summary-csv', str(BICUBIC_VAL_SUMMARY),
])

# 2. Unmodified Base DoSSR (no LoRA) on Real-CE Val (10 seeds, fixed noise per image-seed)
BASE_DOSSR_RES = eval_study_on_val(
    'base_dossr',
    REPO_DIR / 'configs' / 'base_dossr_realce_val.yaml',
    study_out=OUT_BASE / 'base-dossr-realce-val-10seed',
    lora_weights=None,
)

## CELL 04 — Sweep $\lambda_{\text{cons}} \in \{0.1, 0.2\}$ (`400` Steps = `1,600` Minibatches, `seed=231`, `--verify-fp32-grad`) & Compare Against `0.0` / `0.5` / `Base DoSSR` / `Bicubic`

In [ ]:
# === CELL 04: Train lambda=0.1 and lambda=0.2 (400 steps, seed=231), evaluate on val, & compare ===
L3_01_TRAIN = OUT_BASE / 'lora3-lambda01-pilot'
L3_02_TRAIN = OUT_BASE / 'lora3-lambda02-pilot'

run_streamed([
    sys.executable, '-u', 'scripts/run_lora_trial.py',
    '--config', 'configs/lora3_lambda01_pilot.yaml',
    '--train-manifest', 'data/realce_train_manifest.csv',
    '--output-dir', str(L3_01_TRAIN),
    '--verify-fp32-grad',
])

run_streamed([
    sys.executable, '-u', 'scripts/run_lora_trial.py',
    '--config', 'configs/lora3_lambda02_pilot.yaml',
    '--train-manifest', 'data/realce_train_manifest.csv',
    '--output-dir', str(L3_02_TRAIN),
])

L3_01_RES = eval_study_on_val(
    'lora3_lam01',
    REPO_DIR / 'configs' / 'lora3_lambda01_pilot.yaml',
    study_out=OUT_BASE / 'pilot-lora3-lambda01-realce-val-10seed',
    lora_weights=L3_01_TRAIN / 'lora_weights_final.pt',
)
L3_02_RES = eval_study_on_val(
    'lora3_lam02',
    REPO_DIR / 'configs' / 'lora3_lambda02_pilot.yaml',
    study_out=OUT_BASE / 'pilot-lora3-lambda02-realce-val-10seed',
    lora_weights=L3_02_TRAIN / 'lora_weights_final.pt',
)

# Build unified comparison table across Base DoSSR, Pilot LoRA2 (0.0), LoRA3 (0.1, 0.2, 0.5)
run_streamed([
    sys.executable, '-u', 'scripts/analyze_realce_lora_sweep.py',
    '--method', f"Base DoSSR={BASE_DOSSR_RES['ocr_dir']}:{BASE_DOSSR_RES['metrics_summary']}",
    '--method', 'LoRA2 (lam=0.0, 400st)=results/pilot_lora2_vs_lora3_reconly:results/metrics_pilot_lora2_val_summary.csv',
    '--method', f"LoRA3 (lam=0.1, 400st)={L3_01_RES['ocr_dir']}:{L3_01_RES['metrics_summary']}:{L3_01_TRAIN / 'lora_run_manifest.json'}",
    '--method', f"LoRA3 (lam=0.2, 400st)={L3_02_RES['ocr_dir']}:{L3_02_RES['metrics_summary']}:{L3_02_TRAIN / 'lora_run_manifest.json'}",
    '--method', 'LoRA3 (lam=0.5, 400st)=results/pilot_lora3_reconly:results/metrics_pilot_lora3_val_summary.csv',
    '--output-json', str(VAL_RESULTS_DIR / 'phase2_lambda_sweep_comparison.json'),
])

## CELL 05 — Timestep-Restricted Consistency Ablation (`t <= 499`, matching `trunc_12` $t_1 = 500$)

In [ ]:
# === CELL 05: Set SELECTED_LAMBDA after inspecting CELL 04 output (default 0.2), then test cons_t_max=499 ===
SELECTED_LAMBDA = 0.2  # <-- Adjust to 0.1 or 0.2 based on CELL 04 validation table

L3_T499_TRAIN = OUT_BASE / 'lora3-t499-pilot'
run_streamed([
    sys.executable, '-u', 'scripts/run_lora_trial.py',
    '--config', 'configs/lora3_t499_pilot.yaml',
    '--train-manifest', 'data/realce_train_manifest.csv',
    '--lambda-cons', str(SELECTED_LAMBDA),
    '--cons-t-max', '499',
    '--output-dir', str(L3_T499_TRAIN),
])

t_manifest = json.loads((L3_T499_TRAIN / 'lora_run_manifest.json').read_text(encoding='utf-8'))
print('Sampled timestep & active consistency summary:')
print(json.dumps(t_manifest['timestep_and_consistency_stats'], indent=2))

L3_T499_RES = eval_study_on_val(
    f'lora3_lam{SELECTED_LAMBDA}_t499',
    REPO_DIR / 'configs' / 'lora3_t499_pilot.yaml',
    study_out=OUT_BASE / f'pilot-lora3-t499-realce-val-10seed',
    lora_weights=L3_T499_TRAIN / 'lora_weights_final.pt',
)

unrestricted_res = L3_01_RES if SELECTED_LAMBDA == 0.1 else L3_02_RES
run_streamed([
    sys.executable, '-u', 'scripts/analyze_realce_lora_sweep.py',
    '--method', f"Base DoSSR={BASE_DOSSR_RES['ocr_dir']}:{BASE_DOSSR_RES['metrics_summary']}",
    '--method', 'LoRA2 (lam=0.0, 400st)=results/pilot_lora2_vs_lora3_reconly:results/metrics_pilot_lora2_val_summary.csv',
    '--method', f"LoRA3 (lam={SELECTED_LAMBDA}, full_t)={unrestricted_res['ocr_dir']}:{unrestricted_res['metrics_summary']}",
    '--method', f"LoRA3 (lam={SELECTED_LAMBDA}, t<=499)={L3_T499_RES['ocr_dir']}:{L3_T499_RES['metrics_summary']}:{L3_T499_TRAIN / 'lora_run_manifest.json'}",
    '--output-json', str(VAL_RESULTS_DIR / 'phase2_timestep_gating_comparison.json'),
])

## CELL 06 — Scale Up LoRA 2 & Selected LoRA 3 to `2,000` Steps (`save_every=400`), Checkpoint Curves, & Optional 2nd Seed (`seed=777`)

In [ ]:
# === CELL 06: 2,000-step full training (8,000 minibatches) + intermediate checkpoint validation curve ===
FINAL_LAMBDA = SELECTED_LAMBDA
FINAL_CONS_T_MAX = 499  # Set to 999 if unrestricted t=[0,999] outperformed t<=499 in CELL 05
TRAIN_SEEDS = [231]     # Change to [231, 777] to repeat paired comparison with a second independent seed
CKPT_STEPS = [400, 800, 1200, 1600, 2000]

for tr_seed in TRAIN_SEEDS:
    l2_dir = OUT_BASE / f'lora2-full2000-s{tr_seed}'
    l3_dir = OUT_BASE / f'lora3-full2000-s{tr_seed}'
    run_streamed([
        sys.executable, '-u', 'scripts/run_lora_trial.py',
        '--config', 'configs/lora2_full2000_s231.yaml',
        '--train-manifest', 'data/realce_train_manifest.csv',
        '--seed', str(tr_seed),
        '--output-dir', str(l2_dir),
    ])
    run_streamed([
        sys.executable, '-u', 'scripts/run_lora_trial.py',
        '--config', 'configs/lora3_full2000_s231.yaml',
        '--train-manifest', 'data/realce_train_manifest.csv',
        '--seed', str(tr_seed),
        '--lambda-cons', str(FINAL_LAMBDA),
        '--cons-t-max', str(FINAL_CONS_T_MAX),
        '--output-dir', str(l3_dir),
    ])

    curve_args = [
        '--method', f"Base DoSSR={BASE_DOSSR_RES['ocr_dir']}:{BASE_DOSSR_RES['metrics_summary']}",
    ]
    for variant_tag, train_dir, base_yaml in [
        ('lora2', l2_dir, REPO_DIR / 'configs' / 'lora2_full2000_s231.yaml'),
        ('lora3', l3_dir, REPO_DIR / 'configs' / 'lora3_full2000_s231.yaml'),
    ]:
        for st in CKPT_STEPS:
            w_path = train_dir / ('lora_weights_final.pt' if st == 2000 else f'lora_weights_step_{st:04d}.pt')
            tag = f'{variant_tag}_s{tr_seed}_st{st:04d}'
            res = eval_study_on_val(
                tag,
                base_yaml,
                study_out=OUT_BASE / f'val-{tag}-10seed',
                lora_weights=w_path,
            )
            m_arg = f"{train_dir / 'lora_run_manifest.json'}" if st == 2000 else ''
            curve_args.extend(['--method', f"{tag}={res['ocr_dir']}:{res['metrics_summary']}:{m_arg}"])

    run_streamed([
        sys.executable, '-u', 'scripts/analyze_realce_lora_sweep.py',
        *curve_args,
        '--output-json', str(VAL_RESULTS_DIR / f'phase2_2000step_curves_s{tr_seed}.json'),
    ])